In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rag.reranker import Reranker

reranker = Reranker()

INITIALIZING CROSS-ENCODER RERANKER

Loading model:
cross-encoder/ms-marco-MiniLM-L-6-v2


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


Reranker loaded successfully.


In [3]:
import sys
from pathlib import Path

# ============================================================
# PROJECT ROOT
# ============================================================

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:")
print(PROJECT_ROOT)


# ============================================================
# IMPORT HYBRID RETRIEVER
# ============================================================

from rag.hybrid_retriever import HybridRetriever

print("\nInitializing Hybrid Retriever...")

hybrid = HybridRetriever()

print("\nHybrid Retriever ready!")
print(hybrid.__dict__.keys())

Project root:
C:\Users\User\RAG SYS\Question_Generation

Initializing Hybrid Retriever...
INITIALIZING HYBRID RETRIEVER

Loading dense retriever...
Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded.
Embedding dimension: 384
Loading FAISS index...
Loading metadata...

RETRIEVER INITIALIZED
Vectors: 2150
Metadata: 2150
Dimension: 384
Top-K: 20
Dense retriever ready.

Loading BM25 retriever...
INITIALIZING BM25

Loading question data...
Records loaded: 2150
Questions extracted: 2150

Building BM25 index...
BM25 index built successfully.

BM25 READY
BM25 retriever ready.

HYBRID RETRIEVER READY
Dense top-k : 20
BM25 top-k  : 20
RRF k       : 60

Hybrid Retriever ready!
dict_keys(['dense_top_k', 'bm25_top_k', 'rrf_k', 'dense_retriever', 'bm25_retriever'])


In [4]:
query = "Explain object detection in computer vision."

# Get more candidates before reranking
hybrid_results = hybrid.retrieve(
    query,
    top_k=20
)

# Rerank those candidates
reranked_results = reranker.rerank(
    query,
    hybrid_results,
    top_k=5
)

print("\n" + "=" * 80)
print("HYBRID + CROSS-ENCODER")
print("=" * 80)

print("QUERY:", query)

for rank, result in enumerate(reranked_results, 1):

    metadata = result["metadata"]

    if "metadata" in metadata:
        metadata = metadata["metadata"]

    print(
        f"\n{rank}. "
        f"[Reranker: {result['reranker_score']:.4f}] "
        f"{result['id']}"
    )

    print(
        f"   RRF: {result['score']:.6f}"
    )

    print(
        f"   Question: {metadata['question']}"
    )


HYBRID + CROSS-ENCODER
QUERY: Explain object detection in computer vision.

1. [Reranker: 1.3231] ai_ml_computer_vision_q021
   RRF: 0.031778
   Question: What is object detection?

2. [Reranker: -2.6531] ai_ml_computer_vision_q022
   RRF: 0.028787
   Question: What is the difference between image classification and object detection?

3. [Reranker: -2.7315] ai_ml_computer_vision_q023
   RRF: 0.031010
   Question: What is a bounding box in object detection?

4. [Reranker: -3.6952] ai_ml_computer_vision_q038
   RRF: 0.014085
   Question: What is YOLO and why is it widely used for object detection?

5. [Reranker: -3.7190] ai_ml_computer_vision_q025
   RRF: 0.026905
   Question: What is Non-Maximum Suppression and why is it used in object detection?


In [7]:
# ============================================================
# HYBRID + CROSS-ENCODER EVALUATION
# USING EXISTING 200 RETRIEVAL QUERIES
# ============================================================

import json
from pathlib import Path

# ------------------------------------------------------------
# PATH
# ------------------------------------------------------------

EVAL_FILE = PROJECT_ROOT / "tests" / "retrieval_eval.json"

print("=" * 75)
print("HYBRID + CROSS-ENCODER EVALUATION")
print("=" * 75)

print("\nEvaluation file:")
print(EVAL_FILE)

print("File exists:", EVAL_FILE.exists())

if not EVAL_FILE.exists():
    raise FileNotFoundError(f"Could not find:\n{EVAL_FILE}")


# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

with open(EVAL_FILE, "r", encoding="utf-8") as f:
    evaluation_data = json.load(f)

print("Evaluation queries:", len(evaluation_data))


# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

recall_at_1 = 0
recall_at_3 = 0
recall_at_5 = 0

reciprocal_ranks = []

failed = []


# ------------------------------------------------------------
# EVALUATION
# ------------------------------------------------------------

total = len(evaluation_data)

for i, item in enumerate(evaluation_data, 1):

    query = item["query"]
    expected_id = item["expected_id"]

    # ========================================================
    # STEP 1 — HYBRID RETRIEVAL
    # ========================================================

    hybrid_results = hybrid.retrieve(
        query,
        top_k=20
    )

    # ========================================================
    # STEP 2 — CROSS-ENCODER RERANKER
    # ========================================================

    reranked_results = reranker.rerank(
        query,
        hybrid_results,
        top_k=5
    )

    retrieved_ids = [
        result["id"]
        for result in reranked_results
    ]

    # ========================================================
    # RECALL@1
    # ========================================================

    if expected_id in retrieved_ids[:1]:
        recall_at_1 += 1

    # ========================================================
    # RECALL@3
    # ========================================================

    if expected_id in retrieved_ids[:3]:
        recall_at_3 += 1

    # ========================================================
    # RECALL@5
    # ========================================================

    if expected_id in retrieved_ids[:5]:
        recall_at_5 += 1

    # ========================================================
    # MRR
    # ========================================================

    if expected_id in retrieved_ids:

        rank = retrieved_ids.index(expected_id) + 1

        reciprocal_ranks.append(1 / rank)

    else:

        reciprocal_ranks.append(0)

        failed.append({
            "query": query,
            "expected_id": expected_id,
            "retrieved_ids": retrieved_ids
        })

    # ========================================================
    # PROGRESS
    # ========================================================

    if i % 20 == 0 or i == total:

        print(
            f"Processed {i}/{total} "
            f"({i / total * 100:.1f}%)"
        )


# ------------------------------------------------------------
# FINAL METRICS
# ------------------------------------------------------------

recall_1 = recall_at_1 / total
recall_3 = recall_at_3 / total
recall_5 = recall_at_5 / total

mrr = sum(reciprocal_ranks) / total


# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

print("\n")
print("=" * 75)
print("HYBRID + CROSS-ENCODER FINAL RESULTS")
print("=" * 75)

print(f"\nTotal queries: {total}")

print(f"Recall@1: {recall_1 * 100:.2f}%")
print(f"Recall@3: {recall_3 * 100:.2f}%")
print(f"Recall@5: {recall_5 * 100:.2f}%")
print(f"MRR:      {mrr:.4f}")

print(f"\nFailed @5: {len(failed)}")

print("\n" + "=" * 75)

HYBRID + CROSS-ENCODER EVALUATION

Evaluation file:
C:\Users\User\RAG SYS\Question_Generation\tests\retrieval_eval.json
File exists: True
Evaluation queries: 200
Processed 20/200 (10.0%)
Processed 40/200 (20.0%)
Processed 60/200 (30.0%)
Processed 80/200 (40.0%)
Processed 100/200 (50.0%)
Processed 120/200 (60.0%)
Processed 140/200 (70.0%)
Processed 160/200 (80.0%)
Processed 180/200 (90.0%)
Processed 200/200 (100.0%)


HYBRID + CROSS-ENCODER FINAL RESULTS

Total queries: 200
Recall@1: 97.00%
Recall@3: 100.00%
Recall@5: 100.00%
MRR:      0.9842

Failed @5: 0



In [8]:
# ============================================================
# HYBRID + CROSS-ENCODER
# FULL 2,150 QUESTION EVALUATION
# ============================================================

print("=" * 75)
print("HYBRID + CROSS-ENCODER — FULL 2,150 EVALUATION")
print("=" * 75)

# ------------------------------------------------------------
# Get questions from the BM25 retriever
# ------------------------------------------------------------

questions = hybrid.bm25_retriever.questions
metadata = hybrid.bm25_retriever.metadata

total = len(questions)

print(f"\nTotal questions: {total}")

if total != 2150:
    print(f"WARNING: Expected 2150 questions, but found {total}")


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

recall_at_1 = 0
recall_at_3 = 0
recall_at_5 = 0

reciprocal_ranks = []

failed_at_5 = []


# ------------------------------------------------------------
# Evaluate every question
# ------------------------------------------------------------

for i, query in enumerate(questions, 1):

    # The correct answer is the question itself.
    expected_id = metadata[i - 1]["id"]

    # --------------------------------------------------------
    # Hybrid retrieval
    # --------------------------------------------------------

    hybrid_results = hybrid.retrieve(
        query,
        top_k=20
    )

    # --------------------------------------------------------
    # Cross-encoder reranking
    # --------------------------------------------------------

    reranked_results = reranker.rerank(
        query,
        hybrid_results,
        top_k=5
    )

    retrieved_ids = [
        result["id"]
        for result in reranked_results
    ]

    # --------------------------------------------------------
    # Recall@1
    # --------------------------------------------------------

    if expected_id in retrieved_ids[:1]:
        recall_at_1 += 1

    # --------------------------------------------------------
    # Recall@3
    # --------------------------------------------------------

    if expected_id in retrieved_ids[:3]:
        recall_at_3 += 1

    # --------------------------------------------------------
    # Recall@5
    # --------------------------------------------------------

    if expected_id in retrieved_ids[:5]:
        recall_at_5 += 1

    # --------------------------------------------------------
    # MRR
    # --------------------------------------------------------

    if expected_id in retrieved_ids:

        rank = retrieved_ids.index(expected_id) + 1

        reciprocal_ranks.append(1 / rank)

    else:

        reciprocal_ranks.append(0)

        failed_at_5.append({
            "query": query,
            "expected_id": expected_id,
            "retrieved_ids": retrieved_ids
        })

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if i % 100 == 0 or i == total:

        print(
            f"Processed {i}/{total} "
            f"({i / total * 100:.1f}%)"
        )


# ------------------------------------------------------------
# Calculate final metrics
# ------------------------------------------------------------

recall_1 = recall_at_1 / total
recall_3 = recall_at_3 / total
recall_5 = recall_at_5 / total

mrr = sum(reciprocal_ranks) / total


# ------------------------------------------------------------
# Final results
# ------------------------------------------------------------

print("\n")
print("=" * 75)
print("HYBRID + CROSS-ENCODER — FULL RESULTS")
print("=" * 75)

print(f"\nTotal questions: {total}")

print(f"Recall@1: {recall_1 * 100:.2f}%")
print(f"Recall@3: {recall_3 * 100:.2f}%")
print(f"Recall@5: {recall_5 * 100:.2f}%")
print(f"MRR:      {mrr:.4f}")

print(f"\nFailed @5: {len(failed_at_5)}")

print("=" * 75)

HYBRID + CROSS-ENCODER — FULL 2,150 EVALUATION

Total questions: 2150
Processed 100/2150 (4.7%)
Processed 200/2150 (9.3%)
Processed 300/2150 (14.0%)
Processed 400/2150 (18.6%)
Processed 500/2150 (23.3%)
Processed 600/2150 (27.9%)
Processed 700/2150 (32.6%)
Processed 800/2150 (37.2%)
Processed 900/2150 (41.9%)
Processed 1000/2150 (46.5%)
Processed 1100/2150 (51.2%)
Processed 1200/2150 (55.8%)
Processed 1300/2150 (60.5%)
Processed 1400/2150 (65.1%)
Processed 1500/2150 (69.8%)
Processed 1600/2150 (74.4%)
Processed 1700/2150 (79.1%)
Processed 1800/2150 (83.7%)
Processed 1900/2150 (88.4%)
Processed 2000/2150 (93.0%)
Processed 2100/2150 (97.7%)
Processed 2150/2150 (100.0%)


HYBRID + CROSS-ENCODER — FULL RESULTS

Total questions: 2150
Recall@1: 98.19%
Recall@3: 100.00%
Recall@5: 100.00%
MRR:      0.9907

Failed @5: 0


In [10]:
# ============================================================
# TOP 20 HYBRID CANDIDATES — ROBUST DISPLAY
# ============================================================

query = "Explain object detection in computer vision."

results = hybrid.retrieve(
    query,
    top_k=20
)

print("=" * 80)
print("TOP 20 HYBRID CANDIDATES")
print("=" * 80)

print(f"QUERY: {query}")
print("=" * 80)

for rank, result in enumerate(results, 1):

    # --------------------------------------------------------
    # Get metadata safely
    # --------------------------------------------------------

    metadata = result.get("metadata", {})

    # Some results have:
    # metadata = {"id": ..., "metadata": {...}}
    #
    # Others may have:
    # metadata = {"question": ..., ...}

    if isinstance(metadata, dict) and "metadata" in metadata:
        metadata = metadata["metadata"]

    # --------------------------------------------------------
    # Get question safely
    # --------------------------------------------------------

    question = metadata.get(
        "question",
        result.get("question", "N/A")
    )

    # --------------------------------------------------------
    # Print result
    # --------------------------------------------------------

    print(
        f"\n{rank}. "
        f"[RRF: {result.get('score', 0):.6f}] "
        f"{result.get('id', 'N/A')}"
    )

    print(f"   Question: {question}")

TOP 20 HYBRID CANDIDATES
QUERY: Explain object detection in computer vision.

1. [RRF: 0.031778] ai_ml_computer_vision_q021
   Question: What is object detection?

2. [RRF: 0.031010] ai_ml_computer_vision_q023
   Question: What is a bounding box in object detection?

3. [RRF: 0.029199] ai_ml_computer_vision_q042
   Question: What is the difference between face detection and face recognition?

4. [RRF: 0.028787] ai_ml_computer_vision_q022
   Question: What is the difference between image classification and object detection?

5. [RRF: 0.028382] ai_ml_computer_vision_q002
   Question: What are common applications of Computer Vision?

6. [RRF: 0.027912] ai_ml_computer_vision_q001
   Question: What is Computer Vision and what problems does it solve?

7. [RRF: 0.026905] ai_ml_computer_vision_q025
   Question: What is Non-Maximum Suppression and why is it used in object detection?

8. [RRF: 0.025487] ai_ml_computer_vision_q039
   Question: What is the difference between one-stage and two-stag

In [11]:
# ============================================================
# MANUAL SEMANTIC RETRIEVAL TEST
# ============================================================

test_queries = [
    "How can a computer vision model locate objects in an image?",
    "How do neural networks learn useful representations?",
    "What is the purpose of non maximum suppression?",
    "How does Kubernetes handle containers?",
    "What does a REST API do?",
    "How does a relational database store information?",
    "What is the difference between supervised and unsupervised learning?",
    "How does a convolutional neural network process images?",
    "What is transfer learning?",
    "How does Docker isolate applications?",
    "What is the role of an API endpoint?",
    "How does object classification differ from detecting objects?",
    "What is overfitting in machine learning?",
    "How does gradient descent optimize a neural network?",
    "What is the purpose of normalization in machine learning?",
    "How does face recognition work?",
    "What is image segmentation used for?",
    "How does a neural network avoid learning too much noise?",
    "What is the difference between TCP and UDP?",
    "How does a cloud storage system work?"
]

print("=" * 80)
print("MANUAL SEMANTIC RETRIEVAL TEST")
print("=" * 80)

for i, query in enumerate(test_queries, 1):

    hybrid_results = hybrid.retrieve(
        query,
        top_k=20
    )

    reranked_results = reranker.rerank(
        query,
        hybrid_results,
        top_k=5
    )

    print(f"\n{'=' * 80}")
    print(f"TEST {i}")
    print(f"QUERY: {query}")
    print("-" * 80)

    for rank, result in enumerate(reranked_results, 1):

        metadata = result.get("metadata", {})

        if isinstance(metadata, dict) and "metadata" in metadata:
            metadata = metadata["metadata"]

        question = metadata.get(
            "question",
            result.get("question", "N/A")
        )

        print(
            f"{rank}. "
            f"[{result.get('reranker_score', 0):.4f}] "
            f"{result.get('id', 'N/A')}"
        )

        print(f"   {question}")

MANUAL SEMANTIC RETRIEVAL TEST

TEST 1
QUERY: How can a computer vision model locate objects in an image?
--------------------------------------------------------------------------------
1. [-2.7019] ai_ml_computer_vision_q049
   What factors should you consider when deploying a computer vision model?
2. [-3.3785] ai_ml_computer_vision_q002
   What are common applications of Computer Vision?
3. [-3.5879] ai_ml_computer_vision_q050
   Describe how you would build an end-to-end computer vision project from raw images to deployment.
4. [-3.9005] ai_ml_computer_vision_q048
   How would you design a computer vision system that needs real-time inference?
5. [-4.0887] ai_ml_computer_vision_q044
   How can data leakage occur in a computer vision dataset?

TEST 2
QUERY: How do neural networks learn useful representations?
--------------------------------------------------------------------------------
1. [-4.5340] ai_ml_nlp_q025
   What is a recurrent neural network and why is it useful for seq

In [12]:
# ============================================================
# HARD NEGATIVE TEST
# ============================================================

query = "Explain object detection in computer vision."

hybrid_results = hybrid.retrieve(
    query,
    top_k=20
)

reranked_results = reranker.rerank(
    query,
    hybrid_results,
    top_k=20
)

print("=" * 80)
print("HARD NEGATIVE TEST")
print("=" * 80)

for rank, result in enumerate(reranked_results, 1):

    metadata = result.get("metadata", {})

    if isinstance(metadata, dict) and "metadata" in metadata:
        metadata = metadata["metadata"]

    question = metadata.get(
        "question",
        result.get("question", "N/A")
    )

    print(
        f"{rank:2d}. "
        f"Reranker={result.get('reranker_score', 0):7.4f} | "
        f"RRF={result.get('score', 0):.6f}"
    )

    print(f"    {question}")

HARD NEGATIVE TEST
 1. Reranker= 1.3231 | RRF=0.031778
    What is object detection?
 2. Reranker=-2.6531 | RRF=0.028787
    What is the difference between image classification and object detection?
 3. Reranker=-2.7315 | RRF=0.031010
    What is a bounding box in object detection?
 4. Reranker=-3.6952 | RRF=0.014085
    What is YOLO and why is it widely used for object detection?
 5. Reranker=-3.7190 | RRF=0.026905
    What is Non-Maximum Suppression and why is it used in object detection?
 6. Reranker=-4.0509 | RRF=0.014925
    What is mean Average Precision (mAP) in object detection?
 7. Reranker=-4.6726 | RRF=0.028382
    What are common applications of Computer Vision?
 8. Reranker=-5.0105 | RRF=0.027912
    What is Computer Vision and what problems does it solve?
 9. Reranker=-5.5564 | RRF=0.014706
    How can data leakage occur in a computer vision dataset?
10. Reranker=-5.7798 | RRF=0.025487
    What is the difference between one-stage and two-stage object detectors?
11. Rerank

In [13]:
# ============================================================
# SHORT QUERY TEST
# ============================================================

short_queries = [
    "object detection",
    "CNN",
    "Kubernetes",
    "Docker",
    "overfitting",
    "REST API",
    "face recognition",
    "image segmentation",
    "gradient descent",
    "TCP handshake"
]

for query in short_queries:

    results = hybrid.retrieve(query, top_k=20)

    reranked = reranker.rerank(
        query,
        results,
        top_k=3
    )

    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    for rank, result in enumerate(reranked, 1):

        metadata = result.get("metadata", {})

        if isinstance(metadata, dict) and "metadata" in metadata:
            metadata = metadata["metadata"]

        print(
            f"{rank}. "
            f"{result.get('id', 'N/A')} — "
            f"{metadata.get('question', 'N/A')}"
        )


QUERY: object detection
1. ai_ml_computer_vision_q021 — What is object detection?
2. ai_ml_computer_vision_q022 — What is the difference between image classification and object detection?
3. ai_ml_computer_vision_q023 — What is a bounding box in object detection?

QUERY: CNN
1. ai_ml_deep_learning_q076 — Why are CNNs generally more suitable than fully connected networks for images?
2. ai_ml_computer_vision_q017 — What are the typical components of a CNN architecture?
3. ai_ml_deep_learning_q048 — What is a feature map in a CNN?

QUERY: Kubernetes
1. devops_kubernetes_q050 — Design a production Kubernetes platform for a system containing a frontend, backend API, database, background workers, and machine learning model serving.
2. devops_kubernetes_q049 — Design a Kubernetes architecture for a highly available web application.
3. devops_monitoring_q050 — Design a complete monitoring and observability architecture for a Kubernetes-based application with frontend, backend API, database, b